# Materials Science QLoRA Fine-Tuning — Qwen2.5-7B-Instruct

**Goal:** Fine-tune Qwen2.5-7B-Instruct on 100 parsed materials science documents using QLoRA.  
**Tasks trained:** Property Q&A, JSON extraction from text, chatbot-style conversations.  
**GPU:** Kaggle T4 (16GB VRAM) — 4-bit NF4 quantization + gradient checkpointing.  
**Output:** LoRA adapters (~50-100MB) saved to `/kaggle/working/`.  

---

## 1. Setup & Installs

In [ ]:
import torch
import json
import glob
import random
import gc
import os
import re
import time
from pathlib import Path
from collections import Counter

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TextStreamer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

**Edit `DATA_DIR` below** to match your Kaggle dataset path.

In [ ]:
# ── Auto-detect environment ──────────────────────────────────────────────
import platform

IS_KAGGLE = os.path.exists("/kaggle")
IS_WINDOWS = platform.system() == "Windows"

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/matsci-parsed-docs/json_exports"
    OUTPUT_DIR = "/kaggle/working/lora-checkpoints"
    ADAPTER_DIR = "/kaggle/working/lora-adapters"
else:
    # Local: hardcoded path as fallback
    DATA_DIR = r"E:\rlresearchassistant\kaggle_data\json_exports"
    OUTPUT_DIR = os.path.join(os.getcwd(), "lora-checkpoints")
    ADAPTER_DIR = os.path.join(os.getcwd(), "lora-adapters")

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} ({'Windows' if IS_WINDOWS else 'Linux/Mac'})")
print(f"Data dir:    {DATA_DIR}")
print(f"Data dir exists: {os.path.isdir(DATA_DIR)}")
print(f"JSON files:  {len(glob.glob(os.path.join(DATA_DIR, '*.json')))}")

assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR} — edit the path above!"
assert len(glob.glob(os.path.join(DATA_DIR, "*.json"))) > 0, f"No .json files in {DATA_DIR}!"

# Model
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 1024
WARMUP_RATIO = 0.05

# Seed
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Create output dirs
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

## 3. Load Parsed JSON Documents

In [14]:
json_files = sorted(glob.glob(f"{DATA_DIR}/*.json"))
print(f"Found {len(json_files)} JSON files")

documents = []
for f in json_files:
    with open(f, "r", encoding="utf-8") as fh:
        documents.append(json.load(fh))

# Stats
type_counts = Counter(d.get("doc_type", "unknown") for d in documents)
avg_props = sum(len(d.get("properties", [])) for d in documents) / max(len(documents), 1)
has_text = sum(1 for d in documents if d.get("source_text", "").strip())

print(f"\nDocument types: {dict(type_counts)}")
print(f"Avg properties per doc: {avg_props:.1f}")
print(f"Docs with source text: {has_text}/{len(documents)}")
print(f"\nSample material names:")
for d in documents[:5]:
    print(f"  - {d.get('material_name', 'N/A')} ({d.get('doc_type', '?')}, {len(d.get('properties', []))} props)")

Found 0 JSON files

Document types: {}
Avg properties per doc: 0.0
Docs with source text: 0/0

Sample material names:


## 4. Generate Training Data

Three task types — all deterministic (no LLM needed):  
1. **Property Q&A** (~40%) — factual questions about material properties  
2. **JSON Extraction** (~30%) — extract structured JSON from raw text  
3. **Chatbot / Conversational** (~30%) — summaries, comparisons, recommendations

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior Materials Science Expert at Planet Material Labs with deep expertise in polymer science, composites, metals, and ceramics.\n\n"
    "Use the following document context to answer the question about materials science.\n\n"
    "Guidelines:\n"
    "- Provide detailed technical analysis with specific values and units\n"
    "- Compare materials across grades when multiple are relevant\n"
    "- Explain the science behind the properties\n"
    "- Reference standards (ISO, ASTM, UL) where applicable\n"
    "- Suggest alternative grades or materials when appropriate\n"
    "- If data is missing from the context, state so clearly\n"
    "- Be concise and direct. Answer the specific question asked."
)

# ── Q&A Templates ─────────────────────────────────────────────────────────

PROPERTY_Q_TEMPLATES = [
    "What is the {prop} of {mat}?",
    "Report the {prop} for {mat}.",
    "What value was measured for the {prop} of {mat}?",
    "Can you tell me the {prop} of {mat}?",
]

PROPERTY_A_TEMPLATES = [
    "The {prop} of {mat} is {val} {unit}.{ctx}",
    "{mat} has a {prop} of {val} {unit}.{ctx}",
    "Based on the data, the {prop} for {mat} is measured at {val} {unit}.{ctx}",
]

METHODOLOGY_Q = [
    "What experimental methods were used to characterize {mat}?",
    "Describe the methodology used to study {mat}.",
    "How was {mat} tested and characterized?",
]

FINDINGS_Q = [
    "What are the main findings about {mat}?",
    "Summarize the key findings from the study of {mat}.",
    "What did the research reveal about {mat}?",
]

OBJECTIVE_Q = [
    "What was the research objective for {mat}?",
    "What was the goal of studying {mat}?",
]

CONDITIONS_Q = [
    "What processing conditions are recommended for {mat}?",
    "Describe the processing parameters for {mat}.",
]


def make_example(instruction, output, context=""):
    return {"instruction": instruction, "context": context, "output": output}


def generate_qa_examples(doc):
    """Type 1: Property Q&A from structured data."""
    examples = []
    mat = doc.get("material_name", "this material") or "this material"
    props = doc.get("properties", [])

    # Property-specific Q&A (up to 5 per doc)
    sampled_props = random.sample(props, min(5, len(props))) if props else []
    for p in sampled_props:
        prop_name = p.get("property_name") or p.get("name", "")
        val = p.get("value", "")
        unit = p.get("unit", "")
        ctx_raw = p.get("context", "")
        if not prop_name or val == "":
            continue
        ctx = f" (Test condition: {ctx_raw})" if ctx_raw else ""
        q = random.choice(PROPERTY_Q_TEMPLATES).format(prop=prop_name, mat=mat)
        a = random.choice(PROPERTY_A_TEMPLATES).format(
            prop=prop_name, mat=mat, val=val, unit=unit, ctx=ctx
        )
        examples.append(make_example(q, a))

    # Methodology
    meth = doc.get("methodology", "")
    if meth and len(meth) > 20:
        q = random.choice(METHODOLOGY_Q).format(mat=mat)
        examples.append(make_example(q, meth))

    # Key findings
    findings = doc.get("key_findings", [])
    if findings and isinstance(findings, list) and len(findings) > 0:
        q = random.choice(FINDINGS_Q).format(mat=mat)
        if isinstance(findings[0], dict):
            bullets = "\n".join(f"- {f.get('finding', str(f))}" for f in findings)
        else:
            bullets = "\n".join(f"- {f}" for f in findings)
        a = f"Key findings for {mat}:\n{bullets}"
        examples.append(make_example(q, a))

    # Research objective
    obj = doc.get("research_objective", "")
    if obj and len(obj) > 10:
        q = random.choice(OBJECTIVE_Q).format(mat=mat)
        examples.append(make_example(q, obj))

    # Processing conditions
    conditions = doc.get("processing_conditions", [])
    if conditions and isinstance(conditions, list) and len(conditions) > 0:
        q = random.choice(CONDITIONS_Q).format(mat=mat)
        if isinstance(conditions[0], dict):
            bullets = "\n".join(
                f"- {c.get('name', '')}: {c.get('value', '')}" for c in conditions
            )
        else:
            bullets = "\n".join(f"- {c}" for c in conditions)
        a = f"Processing conditions for {mat}:\n{bullets}"
        examples.append(make_example(q, a))

    return examples


def generate_extraction_examples(doc):
    """Type 2: JSON extraction from source text."""
    examples = []
    source = doc.get("source_text", "")
    if not source or len(source) < 100:
        return examples

    mat = doc.get("material_name", "") or ""
    doc_type = doc.get("doc_type", "tds")

    # Full extraction task
    text_snippet = source[:2000]
    if doc_type == "tds":
        target = {
            "material_name": mat,
            "extraction_confidence": doc.get("extraction_confidence", 0.8),
            "properties": [
                {
                    "name": p.get("property_name") or p.get("name", ""),
                    "value": p.get("value", ""),
                    "unit": p.get("unit", ""),
                    "confidence": p.get("confidence", 0.8),
                    "context": p.get("context", ""),
                }
                for p in doc.get("properties", [])[:10]
            ],
            "processing_conditions": doc.get("processing_conditions", []),
        }
    else:
        target = {
            "extraction_confidence": doc.get("extraction_confidence", 0.8),
            "material_properties_mentioned": [
                {
                    "property": p.get("property_name") or p.get("name", ""),
                    "value": p.get("value", ""),
                    "unit": p.get("unit", ""),
                    "confidence": p.get("confidence", 0.8),
                    "context": p.get("context", ""),
                }
                for p in doc.get("properties", [])[:10]
            ],
            "key_findings": doc.get("key_findings", []),
            "methodology": doc.get("methodology", ""),
            "research_objective": doc.get("research_objective", ""),
        }

    q = f"Extract all material properties from the following {'technical data sheet' if doc_type == 'tds' else 'research paper'} text. Return valid JSON only."
    examples.append(make_example(q, json.dumps(target, indent=2), context=text_snippet))

    # Short extraction: material name + type
    short_text = source[:500]
    short_target = {"material_name": mat, "doc_type": doc_type}
    q2 = "Identify the material name and document type from the following text. Return JSON."
    examples.append(make_example(q2, json.dumps(short_target), context=short_text))

    return examples


def generate_chatbot_examples(doc):
    """Type 3: Chatbot-style conversational examples."""
    examples = []
    mat = doc.get("material_name", "this material") or "this material"
    props = doc.get("properties", [])

    # Summary request
    if props:
        q = f"Give me a complete summary of {mat} and its properties."
        prop_lines = []
        for p in props[:15]:
            pn = p.get("property_name") or p.get("name", "")
            val = p.get("value", "")
            unit = p.get("unit", "")
            prop_lines.append(f"- {pn}: {val} {unit}")

        summary_parts = [f"{mat} is characterized by the following properties:\n"]
        summary_parts.append("\n".join(prop_lines))

        findings = doc.get("key_findings", [])
        if findings:
            summary_parts.append("\nKey findings:")
            for f in (findings[:5] if isinstance(findings, list) else []):
                if isinstance(f, dict):
                    summary_parts.append(f"- {f.get('finding', str(f))}")
                else:
                    summary_parts.append(f"- {f}")

        examples.append(make_example(q, "\n".join(summary_parts)))

    # Application-oriented question
    if props and len(props) >= 3:
        q = f"What applications would {mat} be suitable for based on its properties?"
        top_props = props[:5]
        prop_summary = ", ".join(
            f"{p.get('property_name') or p.get('name', '')}: {p.get('value', '')} {p.get('unit', '')}"
            for p in top_props
        )
        a = (
            f"Based on its measured properties ({prop_summary}), {mat} could be suitable for "
            f"applications requiring these specific material characteristics. The combination of "
            f"these properties suggests potential use in engineering applications where these "
            f"performance metrics are critical."
        )
        examples.append(make_example(q, a))

    return examples


# ── Generate all training data ─────────────────────────────────────────────

all_examples = []
type_counts = Counter()

for doc in documents:
    qa = generate_qa_examples(doc)
    ext = generate_extraction_examples(doc)
    chat = generate_chatbot_examples(doc)

    type_counts["qa"] += len(qa)
    type_counts["extraction"] += len(ext)
    type_counts["chatbot"] += len(chat)

    all_examples.extend(qa)
    all_examples.extend(ext)
    all_examples.extend(chat)

random.shuffle(all_examples)

print(f"Total training examples: {len(all_examples)}")
print(f"  Q&A:         {type_counts['qa']}")
print(f"  Extraction:  {type_counts['extraction']}")
print(f"  Chatbot:     {type_counts['chatbot']}")
if len(all_examples) > 0:
    print(f"\nSample example:")
    print(f"  Q: {all_examples[0]['instruction'][:100]}")
    print(f"  A: {all_examples[0]['output'][:100]}")
else:
    print("\nNo training examples generated — check that JSON documents contain properties, source_text, or other extractable fields.")

## 5. Build Dataset & Train/Val Split

In [ ]:
split_idx = int(len(all_examples) * 0.9)
train_data = all_examples[:split_idx]
val_data = all_examples[split_idx:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train: {len(train_dataset)} examples")
print(f"Val:   {len(val_dataset)} examples")

## 6. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Tokenizer loaded: vocab size = {tokenizer.vocab_size}")

## 7. Load Model (4-bit Quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 does NOT support bf16
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

print(f"Model loaded on {model.device}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 8. Configure LoRA Adapters

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Training Setup

In [ ]:
def tokenize_function(examples):
    """Tokenize training examples using Qwen2.5 ChatML template."""
    all_input_ids = []
    all_attention_masks = []

    for instruction, context, output in zip(
        examples["instruction"], examples["context"], examples["output"]
    ):
        user_msg = instruction
        if context:
            user_msg += f"\n\n{context}"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
        )
        all_input_ids.append(tokenized["input_ids"])
        all_attention_masks.append(tokenized["attention_mask"])

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_input_ids,
    }


# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

print(f"Tokenized train: {len(tokenized_train)} examples")
print(f"Tokenized val:   {len(tokenized_val)} examples")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    max_grad_norm=0.3,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

## 10. Train

In [ ]:
print("Starting training...")
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final train loss: {train_result.training_loss:.4f}")

# Log metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)

# Eval
eval_results = trainer.evaluate()
print(f"Eval loss: {eval_results['eval_loss']:.4f}")

## 11. Save LoRA Adapters

In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check adapter size
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"LoRA adapters saved to: {ADAPTER_DIR}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"\nFiles saved:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"  {f}: {size/1e6:.2f} MB")

## 12. Inference Testing — Automated Queries

In [ ]:
# Use the trained model directly (already has LoRA merged in memory)
model.eval()

test_questions = [
    "What material properties are most important for high-temperature applications?",
    "Summarize the key properties of a carbon fiber reinforced composite.",
    "Extract the material name and properties from this text: 'Nylon 6/6 has a tensile strength of 85 MPa, flexural modulus of 2.8 GPa, and HDT of 75°C at 1.82 MPa.'",
    "What is the difference between tensile strength and flexural strength?",
    "Recommend a material with good electrical conductivity and mechanical strength.",
]


def generate_response(question, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


print("=" * 70)
print("INFERENCE TESTING")
print("=" * 70)

for i, q in enumerate(test_questions, 1):
    print(f"\n--- Question {i} ---")
    print(f"Q: {q}")
    response = generate_response(q)
    print(f"A: {response}")
    print()

## 13. Interactive Chat

Chat with your fine-tuned model. Type `quit` or `exit` to stop.  
Type `clear` to reset conversation history.

In [ ]:
model.eval()

print("=" * 70)
print("INTERACTIVE CHAT — Materials Science Assistant")
print("Commands: 'quit'/'exit' to stop, 'clear' to reset history")
print("=" * 70)

conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

while True:
    user_input = input("\nYou: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Chat ended.")
        break
    if user_input.lower() == "clear":
        conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("[Conversation history cleared]")
        continue

    # Add user message
    conversation_history.append({"role": "user", "content": user_input})

    # Build prompt from full conversation history
    text = tokenizer.apply_chat_template(
        conversation_history, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Truncate if too long (keep most recent context)
    if inputs["input_ids"].shape[1] > MAX_SEQ_LEN - 256:
        print("[Context too long, keeping last few turns]")
        # Keep system + last 4 messages
        conversation_history = conversation_history[:1] + conversation_history[-4:]
        text = tokenizer.apply_chat_template(
            conversation_history, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate with streaming
    print("\nAssistant: ", end="", flush=True)
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            streamer=streamer,
        )

    # Extract and store assistant response
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    conversation_history.append({"role": "assistant", "content": response})

## 14. Cleanup & Summary

In [ ]:
print("=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)
print(f"Model:          {MODEL_NAME}")
print(f"Documents:      {len(documents)}")
print(f"Train examples: {len(train_dataset)}")
print(f"Val examples:   {len(val_dataset)}")
print(f"Epochs:         {EPOCHS}")
print(f"LoRA rank:      {LORA_R}")
print(f"Final train loss: {train_result.training_loss:.4f}")
print(f"Final eval loss:  {eval_results['eval_loss']:.4f}")
print(f"Training time:    {elapsed/60:.1f} min")
print(f"Adapter size:     {adapter_size / 1e6:.1f} MB")
print(f"\nAdapters saved at: {ADAPTER_DIR}")
print(f"\nTo load locally:")
print(f"  from peft import PeftModel")
print(f"  base = AutoModelForCausalLM.from_pretrained('{MODEL_NAME}')")
print(f"  model = PeftModel.from_pretrained(base, '{ADAPTER_DIR}')")
print(f"  model = model.merge_and_unload()  # optional: merge for inference")

# Free GPU memory
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print(f"\nGPU memory freed.")